In [1]:
import json
import os

# P1 Read JSON

In [2]:
# 读取JSON文件
def read_json_file(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
        return data

In [3]:
data_dir = '../../Data/BreastUltrasound/BUV/'

In [4]:
ori_train_json_dir = '../../Data/BreastUltrasound/BUV/'+'imagenet_vid_train_15frames.json'
ori_valid_json_dir = '../../Data/BreastUltrasound/BUV/'+'imagenet_vid_val.json'

In [5]:
ori_train_data = read_json_file(ori_train_json_dir)
ori_valid_data = read_json_file(ori_valid_json_dir)

In [6]:
print(ori_train_data.keys())

dict_keys(['categories', 'videos', 'images', 'annotations'])


In [7]:
ori_train_videos = ori_train_data['videos']
ori_train_images = ori_train_data['images']
ori_train_categories = ori_train_data['categories']
ori_train_annotations = ori_train_data['annotations']

In [8]:
ori_valid_videos = ori_valid_data['videos']
ori_valid_images = ori_valid_data['images']
ori_valid_categories = ori_valid_data['categories']
ori_valid_annotations = ori_valid_data['annotations']

In [9]:
ori_train_videos[1]

{'id': 2,
 'name': 'benign/7de856d2db6d4700',
 'vid_train_frames': [-1,
  11,
  23,
  35,
  47,
  59,
  71,
  83,
  95,
  107,
  119,
  131,
  143,
  155,
  167]}

In [13]:
ori_train_images[1]

{'file_name': 'benign/x28f299ceb056964c/000001.png',
 'height': 670,
 'width': 670,
 'id': 2,
 'frame_id': 1,
 'video_id': 1,
 'is_vid_train_frame': False}

In [9]:
ori_valid_categories[0]

{'id': 1, 'name': 'benign', 'encode_name': 'benign'}

In [10]:
len(ori_train_videos)

149

In [11]:
len(ori_valid_videos)

37

In [12]:
ori_train_images[0]

{'file_name': 'benign/x28f299ceb056964c/000000.png',
 'height': 670,
 'width': 670,
 'id': 1,
 'frame_id': 0,
 'video_id': 1,
 'is_vid_train_frame': False}

总共74+112=186个视频文件夹，符合数据现状

In [13]:
train_image_paths = {}
train_mask_paths = {}
valid_image_paths = {}
valid_mask_paths = {}
for i in range(len(ori_train_images)):
    image_name = ori_train_images[i]['file_name']
    image_path = os.path.join(data_dir,'rawframes', image_name)
    train_image_paths[str(ori_train_images[i]['id'])] = image_path
    mask_path = image_path.replace('.png','_mask.png')
    train_mask_paths[str(ori_train_images[i]['id'])] = mask_path
for i in range(len(ori_valid_images)):
    image_name = ori_valid_images[i]['file_name']
    image_path = os.path.join(data_dir,'rawframes', image_name)
    valid_image_paths[str(ori_valid_images[i]['id'])] = image_path
    mask_path = image_path.replace('.png','_mask.png')
    valid_mask_paths[str(ori_valid_images[i]['id'])] = mask_path

In [14]:
len(train_image_paths.keys()) == len(ori_train_images)

True

In [15]:
len(valid_image_paths.keys()) == len(ori_valid_images)

True

In [16]:
ori_train_images[0]

{'file_name': 'benign/x28f299ceb056964c/000000.png',
 'height': 670,
 'width': 670,
 'id': 1,
 'frame_id': 0,
 'video_id': 1,
 'is_vid_train_frame': False}

In [17]:
train_image_anno_dic = {}
for i in range(len(ori_train_images)):
    train_image_anno_dic[str(ori_train_images[i]['id'])] = []
for i in range(len(ori_train_annotations)):   
    # if ori_train_annotations[i]['image_id'] not in train_image_anno_dic.keys():
    #     train_image_anno_dic[str(ori_train_annotations[i]['image_id'])] = []
    train_image_anno_dic[str(ori_train_annotations[i]['image_id'])].append(ori_train_annotations[i]['bbox'])
    
valid_image_anno_dic = {}
for i in range(len(ori_valid_images)):
    valid_image_anno_dic[str(ori_valid_images[i]['id'])] = []
for i in range(len(ori_valid_annotations)):   
    # if ori_valid_annotations[i]['image_id'] not in valid_image_anno_dic.keys():
    #     valid_image_anno_dic[str(ori_valid_annotations[i]['image_id'])] = []
    valid_image_anno_dic[str(ori_valid_annotations[i]['image_id'])].append(ori_valid_annotations[i]['bbox'])


In [18]:
train_image_anno_dic['12']

[[195, 127, 216, 146]]

In [19]:
len(train_image_anno_dic.keys()) == len(ori_train_images)

True

In [20]:
len(valid_image_anno_dic.keys()) == len(ori_valid_images)

True

In [21]:
len(ori_train_images)

20768

In [22]:
len(ori_valid_images)

4504

# P2 Create MASK

In [4]:
import cv2
import numpy as np
from PIL import Image
import os

In [9]:
def create_rectangle_mask(image_path, mask_path, rectangles):
    if not os.path.exists(image_path):
        print(image_path)
        return
    image = cv2.imread(image_path)
    mask = np.zeros(image.shape[:2], dtype="uint8")
    for i in range(len(rectangles)):
        cv2.rectangle(mask, (rectangles[i][0], rectangles[i][1]), (rectangles[i][0]+rectangles[i][2], rectangles[i][1]+rectangles[i][3]), 255, -1)
        # cv2.imshow("Rectangular Mask", mask)
    cv2.imwrite(mask_path, mask)

In [10]:
create_rectangle_mask('../../Data/BreastUltrasound/BUV/rawframes/malignant/1dc9ca2f1748c2ec/000120.png','test1.png',[[271, 174, 269, 210]])

In [25]:
for i in range(len(ori_train_images)):
    id = str(i+1)
    image_path = train_image_paths[id]
    mask_path = train_mask_paths[id]
    rectangles = train_image_anno_dic[id]
    create_rectangle_mask(image_path, mask_path, rectangles)

In [26]:
for i in range(len(ori_valid_images)):
    id = str(i+1)
    image_path = valid_image_paths[id]
    mask_path = valid_mask_paths[id]
    rectangles = valid_image_anno_dic[id]
    create_rectangle_mask(image_path, mask_path, rectangles)